# R2E Quickstart Guide

Welcome to the R2E (Repository to Environment) quickstart guide! This notebook will walk you through the basic steps of using R2E to convert a GitHub repository into an executable environment, generate equivalence tests, and evaluate code modifications.

## Prerequisites

Before we begin, make sure you have:
1. Installed R2E and its dependencies
2. Installed `r2e-test-server`
3. Started the `r2e-test-server` in a separate terminal with the command: `r2e-test-server start`

In [1]:
import os
import json
import rpyc
from rich.panel import Panel
from rich.text import Text
from rich.text import Text
from rich.align import Align
from rich.markdown import Markdown
from rich.progress import Progress
from rich.table import Table
from rich.layout import Layout

from r2e.utils.data import load_functions, load_functions_under_test
from r2e.paths import EXTRACTED_DATA_DIR, TESTGEN_DIR

from helpers import run_command, summarize, console

## Step 1: Setup and Configuration

First, let's import the necessary modules and set up our configuration.

In [2]:
# Configuration
REPO_URL = "https://github.com/r2e-project/r2e.git"  # You can change this to your desired repo
LOCAL_REPO_PATH = os.path.expanduser("~/buckets/my_repos/r2e")
EXP_ID = "quickstart"
MULTIPROCESS = 8
CACHE_BATCH_SIZE = 100

console.print(Panel(Text("Configuration set!", style="bold green")))

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Configuration set!                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 2: Clone and Setup Repository

Now, let's clone the repository and set it up in R2E's workspace.

In [ ]:
# Clone repository
if not os.path.exists(LOCAL_REPO_PATH):
    os.system(f"git clone {REPO_URL} {LOCAL_REPO_PATH}")
else:
    print(f"Repository already exists at {LOCAL_REPO_PATH}")
console.print(Panel(Text("Repository cloned!", style="bold green")))

# Setup repository in R2E workspace
setup_command = f"python -m r2e.repo_builder.setup_repos --local_repo_path {LOCAL_REPO_PATH}"
output = run_command(setup_command)
console.print(Panel(Text("Repository set up in R2E workspace!", style="bold green")))

## Step 3: Extract Functions and Methods

Next, we'll extract functions and methods from the repository.

In [ ]:
extract_command = f"python -m r2e.repo_builder.extract_func_methods --exp_id {EXP_ID} --overwrite_extracted"
output = run_command(extract_command)
summarize(3, "Extract functions and methods", EXP_ID)

## Step 4: Generate Equivalence Tests

Now, let's generate equivalence tests for the extracted functions and methods.

In [ ]:
generate_command = f"python -m r2e.generators.testgen.generate -i {EXP_ID}_extracted.json --exp_id {EXP_ID} --multiprocess {MULTIPROCESS} --use_cache --cache_batch_size {CACHE_BATCH_SIZE}"
output = run_command(generate_command)
console.print(Panel(Text("Equivalence tests generated!", style="bold green")))

##### Let's look at an example function in the repo: `find_dependency_globals`

In [ ]:
futs = load_functions_under_test(TESTGEN_DIR / f"{EXP_ID}_generate.json")
FUT = [f for f in futs if f.name == "find_dependency_globals"][0]

# show the function under test in markdown python
code = f"""
```python
{FUT.code}
```
"""
console.print(Markdown(code))

##### R2E automatically extracts the *Dependency Slice* for the function under test.

This serves as context for the LLM-based equivalence test generation process. This is a program slice that contains a topologically sorted
list of code elements that the function under test depends on.

In [ ]:
context = f"""{FUT.context.context}"""
console.print(Markdown(context))

##### Now, let's look at the generated equivalence tests for the function `find_dependency_globals`

With these function-test pairs, one can create benchmarks to evaluate the correctness of various LLM4Code tasks such as code generation and code modification.

In [ ]:
code = f"""
```python
{FUT.test_history.latest_tests["test_0"]}
```
"""
console.print(Markdown(code))

## Step 5: Execute Equivalence Tests and Improve Iteratively

Finally, let's execute the equivalence tests and prepare a benchmark.

In [3]:
def get_service():
    conn = rpyc.connect("localhost", 3006)
    return conn.root

def create_exec_query(fut):
    name, file_path = fut.execution_fut_data
    repo_data = json.dumps({"repo_id": None, "repo_path": fut.repo.repo_path})
    function_data = json.dumps({"funclass_names": [name], "file_path": file_path})
    tests = json.dumps({"generated_tests": fut.tests})
    return repo_data, function_data, tests

def init_service(repo_data, function_data, tests):
    service.setup_repo(repo_data)
    service.setup_function(function_data)
    service.setup_test(tests)
    init_response = service.init()
    init_output = str(init_response["output"])
    init_error = str(init_response["error"])
    
    print(init_output)
    print(init_error)
    

# Step 1: Initialize the service
service = get_service()

# Step 2: Setup the repository, function and tests
futs = load_functions_under_test(TESTGEN_DIR / f"{EXP_ID}_generate.json")
FUT = [f for f in futs if f.name == "find_dependency_globals"][0]
repo_data, function_data, tests = create_exec_query(FUT)
init_service(repo_data, function_data, tests)

# Step 3: Execute the equivalence tests
out = service.submit()
summarize(6, "Summarize Execution Results", EXP_ID, execution_output=out)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 6: Summarize Execution Results                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    Test Results Summary                                    Coverage Summary                       
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓         ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓               
┃ Metric      ┃ Value ┃ Test Names                         ┃         ┃ Metric            ┃   Value ┃               
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩         ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩               
│ Total Tests │     6 │                                    │         │ Line Coverage     │ 100.00% │               
│ Passed      │     6 │ test_async_function                │         │ Branch Coverage   │ 100.00% │               
│             │       │ test_class_with_method_and_globals │         │ Executable Lines  │      10 │               
│             │       │ test_function_with_annotations     │         │ Executed Branches │       3 │               
│             │       │ test_function_with_builtins        │         └───────────────────┴─────────┘               
│             │       │ test_simple_function               │                                                       
│             │       │ test_unique_flag                   │                                                       
│ Failed      │     0 │                                    │                                                       
│ Errored     │     0 │                                    │                                                       
└─────────────┴───────┴────────────────────────────────────┘

##### R2E also allows you to improve the tests generated by the LLM iteratively w/ the collected execution feedback.

In [206]:
# TODO: add the code to run test repair here

## Step 6: Beyond code generation benchmarks

R2E generated tests and environments can be used beyond code generation benchmarks. They can also be used for code-to-code modification tasks:
1. Code refactoring
2. Code translation
3. Code optimization
4. Code migration
... and more!

In the example below, we show an example of how to R2E enables reliability for LLM-powered code refactoring.

In [4]:
# Simple example of using GPT-4-o to refactor a functiom
# You can do more complex agent-based interactions too!

from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENAI_KEY"),
)

def extract_markdown(content: str) -> str:
    """Extracts the code block from a markdown file."""
    lines = content.split("\n")
    backtick_indices = [
        i for i, line in enumerate(lines) if line.startswith("```")
    ]
    return "\n".join(lines[backtick_indices[0] + 1 : backtick_indices[1]])

def refactor_code(code):
    PROMPT = (
        f"Refactor the following Python function into 3 smaller functions. Don't generate anything else.\n\n"
        f"```python\n{code}\n```\n\n"
        f"Return the refactored code in a markdown code block."
    )
    output = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a Python programming expert."},
            {"role": "user", "content": PROMPT},
        ],
        model="gpt-4o",
        temperature=0.2,
    )
    refactored_code = extract_markdown(output.choices[0].message.content)
    return refactored_code

refactored_code = refactor_code(FUT.code)
console.print(Markdown(f"```python\n{refactored_code}\n```"))

                                                                                                                   
 import ast                                                                                                        
 import builtins                                                                                                   
                                                                                                                   
 def find_dependency_globals(astnode: ast.stmt, unique: bool=True) -> list[str]:                                   
     """                                                                                                           
     Find all the global symbols in the ast node                                                                   
     Further filters builtins and non-dependency globals                                                           
     :param astnode: ast.AST                                                                                       
     :return: list[str] - list of global symbols                                                                   
     """                                                                                                           
     fake_function_node = create_fake_function(astnode)                                                            
     all_type_annotation_globals = astnode_to_type_annotation_globals(astnode)                                     
     all_globals = get_combined_globals(fake_function_node, all_type_annotation_globals)                           
     filtered_globals = filter_globals(all_globals)                                                                
     if unique:                                                                                                    
         return list(set(filtered_globals))                                                                        
     return filtered_globals                                                                                       
                                                                                                                   
 def get_combined_globals(fake_function_node: ast.AST, type_annotation_globals: list[str]) -> list[str]:           
     """                                                                                                           
     Combine globals from function node and type annotations                                                       
     :param fake_function_node: ast.AST                                                                            
     :param type_annotation_globals: list[str]                                                                     
     :return: list[str]                                                                                            
     """                                                                                                           
     all_globals = get_funclass_globals(fake_function_node)                                                        
     all_globals.extend(type_annotation_globals)                                                                   
     return all_globals                                                                                            
                                                                                                                   
 def filter_globals(all_globals: list[str]) -> list[str]:                                                          
     """                                                                                                           
     Filter out builtins and non-dependency globals                                                                
     :param all_globals: list[str]                                                                                 
     :return: list[str]                                 

##### Let's evaluate the correctness of this refactored code using the generated equivalence tests.

In [5]:
service.setup_codegen_mode()
out = service.execute(refactored_code) # execute the refactored code
out = service.submit() # submit it for equivalence testing
summarize(6, "Summarize Execution Results", EXP_ID, execution_output=out)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 6: Summarize Execution Results                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    Test Results Summary                    
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric      ┃ Value ┃ Test Names                         ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Total Tests │     6 │                                    │
│ Passed      │     6 │ test_async_function                │
│             │       │ test_class_with_method_and_globals │
│             │       │ test_function_with_annotations     │
│             │       │ test_function_with_builtins        │
│             │       │ test_simple_function               │
│             │       │ test_unique_flag                   │
│ Failed      │     0 │                                    │
│ Errored     │     0 │                                    │
└─────────────┴───────┴────────────────────────────────────┘

In [6]:
# break the refactored code by introducing a bug
buggy_code = refactored_code.replace("if unique:", "if not unique:")
buggy_code = buggy_code.replace('print("Executing', 'print("Executing buggy')

service.setup_codegen_mode()
out = service.execute(buggy_code) # execute the buggy code
out = service.submit() # submit it for equivalence testing
summarize(6, "Summarize Execution Results", EXP_ID, execution_output=out)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 6: Summarize Execution Results                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    Test Results Summary                    
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric      ┃ Value ┃ Test Names                         ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Total Tests │     6 │                                    │
│ Passed      │     5 │ test_async_function                │
│             │       │ test_class_with_method_and_globals │
│             │       │ test_function_with_annotations     │
│             │       │ test_function_with_builtins        │
│             │       │ test_simple_function               │
│ Failed      │     1 │ test_unique_flag                   │
│ Errored     │     0 │                                    │
└─────────────┴───────┴────────────────────────────────────┘

## Conclusion

Congratulations! You've now gone through the basic workflow of R2E:
1. Setting up a repository
2. Extracting functions and methods
3. Generating equivalence tests
4. Evaluating a refactored function

This quickstart guide demonstrates the core functionality of R2E. For more advanced features and detailed usage, please refer to our full documentation.